# Foundry Audio Evaluation Demo

Run an audio emotion evaluation in a Microsoft Foundry project with the Foundry SDK 2.x, Microsoft Entra authentication, a Foundry model target, and a cloud evaluation run.

## Prerequisites

- Set `FOUNDRY_PROJECT_ENDPOINT` to a Foundry project endpoint in a region where the selected model deployments are available.
- The project must contain `gpt-audio` and `gpt-4o-mini` deployments (override with `FOUNDRY_AUDIO_MODEL` and `FOUNDRY_GRADER_MODEL`).
- Authenticate with a credential supported by `DefaultAzureCredential`, such as `az login`.
- Install `requirements.txt` in the notebook environment.

## SDK boundary

The notebooks use `AIProjectClient` with `DefaultAzureCredential` for project access and client creation. `azure-ai-projects` 2.7.0 exposes stable `TargetCompletionEvalRunDataSource`, `AzureAIModelTargetParam`, and `ModelSamplingConfigParam` types, which are used for each cloud run.

The current Foundry evaluation create operation is still exposed only through the OpenAI-compatible protocol client returned by `AIProjectClient.get_openai_client()`. The 2.7.0 package does not expose stable Azure types for the custom eval schema, score-model grader, or inline source/input-message members (the stable data-source annotations themselves still reference `openai.types`). Those small payloads are therefore isolated as plain dictionaries in this notebook/helper, with no direct `openai` import, direct client construction, API key, raw route, or account endpoint.


In [ ]:
import os
from dotenv import load_dotenv
from scripts.eval_utils import FoundryEvaluationClient, build_model_target_data_source

load_dotenv()
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
AUDIO_MODEL = os.getenv("FOUNDRY_AUDIO_MODEL", "gpt-audio")
GRADER_MODEL = os.getenv("FOUNDRY_GRADER_MODEL", "gpt-4o-mini")
print(f"Foundry SDK client configuration ready for {PROJECT_ENDPOINT}.")

## Prepare inline audio evaluation data

The helper downloads ten CREMA-D samples, preserves or creates WAV bytes, and writes JSONL records containing base64 audio plus the expected emotion.

In [ ]:
from scripts.audio_utils import load_and_create_audio_dataset, display_items

load_and_create_audio_dataset("AbstractTTS/CREMA-D", max_samples=10)
display_items(3)

In [ ]:
import json

with open("data/audio_emotion_evaluation.jsonl", encoding="utf-8") as f:
    eval_items = [json.loads(line) for line in f]
print(f"Prepared {len(eval_items)} audio evaluation items.")

## Create and run the Foundry cloud evaluation

The score-model grader compares the audio model's one-word emotion prediction with the expected label. The Foundry target data source invokes `gpt-audio` in the cloud; the notebook does not create a separate inference client.

In [ ]:
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {
            "audio_data": {"type": "string", "description": "Base64-encoded WAV audio."},
            "expected_emotion": {"type": "string", "description": "Expected primary emotion."},
        },
        "required": ["audio_data", "expected_emotion"],
    },
    "include_sample_schema": True,
}

score_model_grader = {
    "type": "score_model",
    "name": "audio_emotion_match",
    "model": GRADER_MODEL,
    "input": [
        {
            "role": "system",
            "content": "Compare the predicted emotion with {{item.expected_emotion}}. Return only a float from 0.0 to 1.0; 1.0 is an exact semantic match and 0.0 is incorrect.",
        },
        {
            "role": "user",
            "content": "Expected: {{item.expected_emotion}}\nPrediction: {{sample.output_text}}",
        },
    ],
    "range": [0.0, 1.0],
    "pass_threshold": 0.5,
}

input_messages = {
    "type": "template",
    "template": [
        {
            "role": "system",
            "content": "Identify the primary emotion in the supplied audio.",
        },
        {
            "role": "user",
            "type": "message",
            "content": {
                "type": "input_text",
                "text": "Respond with exactly one word: anger, fear, disgust, happy, neutral, or sad.",
            },
        },
        {
            "role": "user",
            "type": "message",
            "content": {
                "type": "input_audio",
                "input_audio": {"data": "{{item.audio_data}}", "format": "wav"},
            },
        },
    ],
}

data_source = build_model_target_data_source(
    items=eval_items,
    model=AUDIO_MODEL,
    input_messages=input_messages,
    max_completion_tokens=32,
)

In [ ]:
client = FoundryEvaluationClient(PROJECT_ENDPOINT)
try:
    evaluation = client.create_evaluation(
        name="Audio Emotion Evaluation - SDK 2.x",
        data_source_config=data_source_config,
        testing_criteria=[score_model_grader],
    )
    evaluation_run = client.create_run(
        evaluation_id=evaluation.id,
        name="Audio Emotion Evaluation - SDK 2.x",
        data_source=data_source,
        metadata={"demo": "audio", "sdk": "azure-ai-projects-2.7.0"},
    )
    completed_run = client.wait_for_run(
        evaluation_id=evaluation.id,
        run_id=evaluation_run.id,
    )
    output_items = client.list_output_items(
        evaluation_id=evaluation.id,
        run_id=evaluation_run.id,
    )
finally:
    client.close()

## Results

Display grader status/score, expected emotion, and the Foundry target's generated response.

In [ ]:
import pandas as pd

rows = []
for output_item in output_items:
    results = output_item.get("results") or []
    result = results[0] if results else {}
    datasource_item = output_item.get("datasource_item") or {}
    sample = output_item.get("sample") or {}
    sample_output = sample.get("output") or []
    rows.append({
        "id": output_item.get("id"),
        "status": result.get("status"),
        "score": result.get("score"),
        "expected_emotion": datasource_item.get("expected_emotion"),
        "model_response": sample.get("output_text") or (sample_output[0].get("content") if sample_output else None),
    })

results_df = pd.DataFrame(rows)
display(results_df)
print(json.dumps({
    "evaluation_id": evaluation.id,
    "run_id": completed_run.id,
    "status": completed_run.status,
    "report_url": completed_run.report_url,
    "result_counts": completed_run.result_counts.model_dump() if hasattr(completed_run.result_counts, "model_dump") else completed_run.result_counts,
    "output_item_count": len(output_items),
}, indent=2))